# Vision Transformer

**0 前言**<br>
&nbsp;&nbsp;&nbsp;&nbsp;0.1 为什么图像任务也可以使用Transformer？<br>
&nbsp;&nbsp;&nbsp;&nbsp;0.2 从CNN到ViT：视觉建模思路的变化<br>

**1 ViT模型的核心思想与整体结构**<br>
&nbsp;&nbsp;&nbsp;&nbsp;1.1 ViT到底想解决什么问题？<br>
&nbsp;&nbsp;&nbsp;&nbsp;1.2 ViT整体结构与数据流<br>
&nbsp;&nbsp;&nbsp;&nbsp;1.3 ViT和原始Transformer的关系<br>

**2 ViT模型的输入：从图像到Patch序列**<br>
&nbsp;&nbsp;&nbsp;&nbsp;2.1 图像为什么要切成Patch？<br>
&nbsp;&nbsp;&nbsp;&nbsp;2.2 Patch Flatten与Linear Projection<br>
&nbsp;&nbsp;&nbsp;&nbsp;2.3 图像输入在ViT中的张量形状变化<br>

**3 分类标记CLS Token与分类输出**<br>
&nbsp;&nbsp;&nbsp;&nbsp;3.1 为什么要额外加入一个分类标记？<br>
&nbsp;&nbsp;&nbsp;&nbsp;3.2 CLS Token如何汇总整张图像的信息？<br>
&nbsp;&nbsp;&nbsp;&nbsp;3.3 分类头MLP Head如何输出类别概率？<br>

**4 ViT中的位置编码**<br>
&nbsp;&nbsp;&nbsp;&nbsp;4.1 图像Patch为什么也需要位置信息？<br>
&nbsp;&nbsp;&nbsp;&nbsp;4.2 ViT中的可学习位置编码<br>
&nbsp;&nbsp;&nbsp;&nbsp;4.3 位置编码与图像二维结构的关系<br>

**5 ViT中的Transformer Encoder主干**<br>
&nbsp;&nbsp;&nbsp;&nbsp;5.1 ViT中的多头自注意力机制<br>
&nbsp;&nbsp;&nbsp;&nbsp;5.2 ViT中的前馈神经网络MLP Block<br>
&nbsp;&nbsp;&nbsp;&nbsp;5.3 残差连接与Layer Normalization<br>

<font color="red">**6 ViT的训练、特点与局限**</font><br>
&nbsp;&nbsp;&nbsp;&nbsp;6.1 ViT为什么通常需要较大数据量？<br>
&nbsp;&nbsp;&nbsp;&nbsp;6.2 ViT和CNN的归纳偏置差异<br>
&nbsp;&nbsp;&nbsp;&nbsp;6.3 ViT的优势、局限与常见变体<br>


## 6 ViT的训练、特点与局限

到前一章为止，我们已经把ViT的主干结构基本拆完了：

- 图像如何被切成patch
- patch怎样变成token序列
- CLS Token如何汇总整图信息
- 位置编码怎样补上空间位置信息
- Transformer Encoder又怎样通过注意力和MLP不断加工这些表示

到这里，ViT“怎么工作”这件事已经比较清楚了。但如果我们继续从更高一层看，就还会遇到一些特别重要的问题：

- 为什么ViT经常被说“更吃数据”？
- 它和CNN相比，到底有什么本质上的建模差异？
- 它是不是在所有场景下都比CNN更好？

这一章就专门来讨论这些问题。因为理解一个模型，不能只停留在“结构长什么样”，还要知道它适合什么、依赖什么、代价是什么。


### 6.1 ViT为什么通常需要较大数据量？

关于ViT，一个很常见的结论是：**它通常比CNN更依赖大规模数据。**

这不是偶然现象，核心原因可以直接概括成一句话：

**ViT内置的视觉先验更少，所以很多原本在CNN里由结构直接提供的规律，在ViT里要更多依赖数据自己学出来。**

这里说“更需要大数据”，不是说小数据下一定不能用，而是说：

- 从头训练时，ViT通常没有CNN那么容易训好
- 在数据规模不大时，它更容易表现不稳或泛化不占优
- 当数据规模足够大时，它的潜力更容易被释放出来

为什么CNN通常在中小数据上更占便宜？因为CNN天然带有很强的归纳偏置，例如：

- 它默认局部邻域更重要
- 它通过参数共享默认同一种局部模式会在不同位置重复出现
- 它天然适合按“局部到整体”的方式逐层构建语义

这些规律不是靠数据学出来的，而是卷积结构一开始就写进模型里的。所以CNN在样本不多时，往往已经站在了一个更适合视觉任务的起点上。

ViT则不同。虽然它也把图像切成patch，但进入Encoder之后，模型不会天然优先只看局部、也不会天然强调邻近位置必须先交互。它更多依赖注意力和训练数据自己去学：哪些区域重要、哪些空间关系关键、哪些全局模式有助于分类。

这意味着，ViT的自由度更大，但也更需要数据来约束这种自由度。换句话说，**先验越少，模型越需要更多样本来告诉自己应该学什么规律。**

另外，ViT是以patch为单位建模的，而不是像卷积那样从像素邻域开始逐层提取局部模式；再加上它通常模型容量较大，这都会进一步放大它对数据规模和训练策略的依赖。

当然，这并不意味着ViT在小数据上一定不行。只要配合大规模预训练、迁移学习、蒸馏、强数据增强等手段，ViT在较小数据集上也可以表现得很好。只是从原生训练倾向看，它通常比CNN更吃数据。

这一节真正应该记住的是：

- ViT更吃数据，根本原因是视觉先验更少
- CNN先验更强，所以在中小数据上常常更稳
- ViT结构更灵活，但很多规律要靠数据自己学出来
- 因此它通常更依赖大规模数据和更强训练策略

理解了这一点之后，下一步自然就要继续问：**ViT和CNN的归纳偏置到底差在哪？** 这就是下一节 `6.2 ViT和CNN的归纳偏置差异` 要讨论的问题。


**Nano Banana绘图提示词**

画一张教学图，主题是“为什么ViT通常比CNN更需要大数据”。整体采用左右对照布局，白底，中文标签完整，适合深度学习课件。左侧画CNN的学习示意，标注“强归纳偏置：局部感受野、参数共享、逐层聚合”，下方标注“中小数据下更容易学稳”；右侧画ViT的学习示意，标注“结构更通用、先验更少、更多依赖数据学习关系”，下方标注“通常更需要大规模数据”。中间用箭头和对比框总结：`先验强 -> 需要数据相对更少`，`先验少 -> 需要更多数据来学规律`。在底部再加一句总结：“ViT不是不能用小数据，而是原生训练时通常更吃数据”。要求用不同颜色区分CNN和ViT两条路线，整体风格简洁、工整、教学感强。


### 6.2 ViT和CNN的归纳偏置差异

前一节已经讲过，ViT通常更吃数据，一个核心原因就是它和CNN相比，内置的归纳偏置更弱。现在我们把这个差异直接讲清楚。

所谓归纳偏置，简单理解，就是：**模型在看到数据之前，结构本身已经替它预设了哪些规律更值得优先相信。**

在这件事上，CNN和ViT的差别非常鲜明。

CNN的归纳偏置主要体现在：

- 它天然强调局部邻域
- 它天然使用参数共享，默认同一种局部模式可以在不同位置重复出现
- 它通常按“局部到整体”的方式逐层构建语义

这些设计都非常贴合图像，因此CNN一开始就更像是“按视觉任务定制过的结构”。

而ViT的归纳偏置更弱，主要体现在：

- patch进入Encoder后，模型不会天然优先只看局部邻域
- token之间能否建立联系，更多交给注意力和数据自己学习
- 它更像一个通用序列建模框架，而不是专门为图像局部结构定制的网络

所以，两者的差别可以压缩成一句话：

**CNN更像“先把看图像的经验写进结构里”，ViT更像“给你一个更通用的结构，让你从数据里自己学经验”。**

这也直接带来不同的取舍：

- CNN先验更强，因此在中小数据上常常更稳
- ViT先验更弱，但结构更统一、更灵活，在大数据下潜力更大

因此，ViT和CNN的差异不只是“一个用卷积，一个用注意力”，更本质的区别在于：**它们把多少视觉规律写死在结构里，又把多少规律留给数据去学习。**


### 6.3 ViT的优势、局限与常见变体

到这里，我们可以对ViT做一个更完整的收束。

先看它的优势。

ViT的主要优势包括：

- 结构统一，和NLP中的Transformer主干高度一致
- 自注意力天然具备全局建模能力，patch之间可以直接建立远距离联系
- 在大规模数据和预训练条件下，往往能学到很强的表示
- 结构扩展性好，容易和多模态、生成式模型等方向结合

再看它的局限。

ViT的主要局限包括：

- 原生上通常比CNN更依赖大规模数据和较强训练策略
- 对图像二维局部结构的先验较弱
- patch化会带来一定的信息粗粒度化
- 标准自注意力在序列较长时计算和显存代价较高

所以，更准确地说，ViT并不是“全面取代CNN”的简单答案，而是另一条很强的视觉建模路线。它在很多场景里非常成功，但它的成功往往建立在合适的数据规模、训练方案和模型设计之上。

最后简单提一下常见变体。后续很多工作，基本都在围绕ViT的这些短板继续改进，例如：

- 有的工作改进patch表示和局部建模方式
- 有的工作改进位置编码或相对位置建模
- 有的工作降低注意力计算代价
- 有的工作把卷积偏置重新部分引入Transformer

这类变体很多，但它们大多都没有改变ViT最核心的出发点：

**把图像改写成token序列，再用Transformer去建模这些视觉token之间的关系。**

到这里，ViT这组讲义就完成了。你现在应该已经能从输入、位置、CLS Token、Encoder主干、训练特点和局限这些角度，对Vision Transformer形成一条比较完整的理解主线。
